# NeMo Agent Toolkit: Eval Python API Usability

This notebook exercises the eval Python API directly — no CLI — to understand
how easy or painful it is to run evaluation programmatically.

**Goal**: Surface friction points to inform whether a lightweight eval interface is needed.

**Requirements**: Only `nvidia-nat[eval]` and `nvidia-nat-test` — no API keys needed
(we use `nat_test_llm` for deterministic, offline execution).

## 1. Setup: Imports

Let's see how many imports are needed to run eval programmatically.
This is already a usability signal.

In [ ]:
import json
import tempfile
from pathlib import Path

# Core eval runtime
from nat.data_models.evaluate_runtime import EvaluationRunConfig
from nat.plugins.eval.runtime.evaluate import EvaluationRun

# For loading a YAML config (the easier path)
from nat.runtime.loader import load_config

Only 3 imports from NAT to get started. Not bad — but we haven't built a Config
programmatically yet. That's where the friction starts.

## 2. Prepare: Config and Dataset

We'll create a minimal eval config and dataset inline. The config uses `nat_test_llm`
so no API keys or network access are required.

**Note**: We're using YAML here (loaded via `load_config`). Attempting to construct
`Config` as a pure Python object is much more painful — we'll discuss why in the
commentary section.

In [ ]:
# Create a tiny eval dataset
dataset = [
    {"id": 1, "question": "What is 2 + 3?", "answer": "5"},
    {"id": 2, "question": "What is 10 * 4?", "answer": "40"},
    {"id": 3, "question": "What is 100 / 5?", "answer": "20"},
]

tmpdir = Path(tempfile.mkdtemp())
dataset_path = tmpdir / "eval_dataset.json"
dataset_path.write_text(json.dumps(dataset, indent=2))

# Create a minimal YAML config
config_yaml = f"""
llms:
  test_llm:
    _type: nat_test_llm
    response_seq:
      - "The answer is 5"
      - "The answer is 40"
      - "The answer is 20"
    delay_ms: 0

workflow:
  _type: react_agent
  llm_name: test_llm
  tool_names: []
  verbose: false

eval:
  general:
    max_concurrency: 1
    output_dir: {tmpdir / 'output'}
    dataset:
      _type: json
      file_path: {dataset_path}
"""

config_path = tmpdir / "eval_config.yml"
config_path.write_text(config_yaml)

print(f"Config: {config_path}")
print(f"Dataset: {dataset_path} ({len(dataset)} items)")

## 3. Run Evaluation Programmatically

This is the core of what we're testing: can we call eval as a Python API
and get structured results back without file I/O?

In [ ]:
run_config = EvaluationRunConfig(
    config_file=config_path,
    write_output=False,  # No file I/O — pure Python API
)

runner = EvaluationRun(config=run_config)
output = await runner.run_and_evaluate()

print(f"Workflow interrupted: {output.workflow_interrupted}")
print(f"Items evaluated: {len(output.eval_input.eval_input_items)}")
print(f"Evaluators run: {len(output.evaluation_results)}")
print(f"Files written: {output.workflow_output_file}")

**Observation**: 4 lines to run eval and get results. `write_output=False` means
zero disk writes. The `EvaluationRunOutput` has everything we need.

No evaluators were configured (we kept the config minimal), so `evaluation_results`
is empty — but the workflow ran and we have outputs + trajectories.

## 4. Inspect Results

Let's look at what came back — per-item inputs, outputs, and trajectories.

In [ ]:
for item in output.eval_input.eval_input_items:
    print(f"\n--- Item {item.id} ---")
    print(f"  Input:    {item.input_obj}")
    print(f"  Expected: {item.expected_output_obj}")
    print(f"  Output:   {item.output_obj}")
    print(f"  Trajectory events: {len(item.trajectory)}")

## 5. Inspect IntermediateStep Trajectory

This is the key question for ATIF: how ergonomic is the current trace format?
Each item's `trajectory` is a `list[IntermediateStep]`.

In [ ]:
# Inspect the first item's trajectory in detail
item = output.eval_input.eval_input_items[0]
print(f"Item {item.id}: '{item.input_obj}'")
print(f"Trajectory has {len(item.trajectory)} events:\n")

for i, step in enumerate(item.trajectory):
    event_type = step.payload.event_type.value if step.payload and step.payload.event_type else "unknown"
    func_name = step.function_ancestry.function_name if step.function_ancestry else "N/A"

    # Extract input/output from payload
    data_input = None
    data_output = None
    if step.payload and step.payload.data:
        data_input = str(step.payload.data.input)[:80] if step.payload.data.input else None
        data_output = str(step.payload.data.output)[:80] if step.payload.data.output else None

    print(f"  [{i}] {event_type:15s} | func={func_name}")
    if data_input:
        print(f"      input:  {data_input}")
    if data_output:
        print(f"      output: {data_output}")

### Trajectory ergonomics observations

To extract useful information from the trajectory, you need to know:

1. `step.payload.event_type` — enum like `LLM_START`, `LLM_END`, `TOOL_START`, `TOOL_END`
2. `step.payload.data.input` / `step.payload.data.output` — the actual content (deeply nested)
3. `step.function_ancestry.function_name` — which function produced this event
4. `step.parent_id` — for reconstructing the call tree

This is a **flat list** with no hierarchy — you must manually pair START/END events
and reconstruct nesting via `parent_id` + `function_ancestry`.

For simple "what did the LLM say?" inspection, this is workable.
For multi-agent traces, this would be quite painful.

## 6. Why Programmatic Config Construction is Painful

We used a YAML file above. What if we wanted to build Config entirely in Python?
Let's try — this surfaces the biggest friction point.

In [ ]:
# To build a Config programmatically, you'd need to import:
#   from nat.data_models.config import Config
#   from nat.data_models.evaluate_config import EvalConfig, EvalGeneralConfig
#   from nat.data_models.dataset_handler import EvalDatasetJsonConfig
#
# But Config uses pydantic Discriminator validators that require
# all plugins to be discovered and registered first. You can't just do:
#
#   Config(workflow={"_type": "react_agent", "llm_name": "test_llm"})
#
# To make this work, you'd need to:
#   1. Call discover_and_register_plugins() first
#   2. Call Config.rebuild_annotations() to rebuild the discriminated unions
#   3. Then construct Config with raw dicts that match the YAML structure
#
# The easiest programmatic path is load_config() which handles all of this:

config = load_config(config_path)

print(f"Config loaded. Workflow type: {config.workflow.type}")
print(f"LLMs: {list(config.llms.keys())}")
print(f"Dataset: {config.eval.general.dataset}")

# Bottom line: building Config from Python requires understanding the
# plugin discovery + type registry system. The YAML path hides this.
# For a Python API, we need a simpler entry point.

In [ ]:
# Using a Config model directly with EvaluationRunConfig:
run_config_from_model = EvaluationRunConfig(
    config_file=config,   # Pass the Config object directly (no YAML file)
    write_output=False,
)

runner2 = EvaluationRun(config=run_config_from_model)
output2 = await runner2.run_and_evaluate()

print(f"Items: {len(output2.eval_input.eval_input_items)}")
for item in output2.eval_input.eval_input_items:
    print(f"  [{item.id}] {item.input_obj} -> {item.output_obj}")

## 7. Usability Assessment

### What works well

- **Running eval**: 4 lines — `EvaluationRunConfig` + `EvaluationRun` + `await run_and_evaluate()`
- **No file I/O**: `write_output=False` gives clean programmatic access
- **Structured output**: `EvaluationRunOutput` has per-item results, scores, usage stats, trajectories
- **YAML path**: Using a config file is straightforward

### Friction points

1. **Config construction**: Building `Config` in pure Python requires plugin discovery + type registry knowledge. There's no lightweight constructor like `Config.from_dict({...})`.

2. **No single-item eval**: To evaluate one question, you must create a JSON dataset file, configure a `DatasetHandler`, etc. There's no `evaluate_single(question, expected, output)` API.

3. **Trajectory is a flat list**: `IntermediateStep` events are a flat list with event types and parent IDs. Extracting "what tools were called" or "what did the LLM respond" requires manual iteration and pairing of START/END events.

4. **Evaluator coupling**: Evaluators must be registered via the plugin system and configured in YAML/Config. You can't pass a simple `def score(output, expected) -> float` function.

5. **Async-only**: `run_and_evaluate` is async — fine for notebooks but adds friction for simple scripts.

### What a lightweight API could look like

```python
# Hypothetical lightweight interface
from nat.eval import evaluate

# Option A: Evaluate a pre-collected trajectory (offline scoring)
result = evaluate(
    trajectory=atif_trajectory,  # ATIF trajectory object
    evaluators=["accuracy", "latency", my_custom_fn],
)
print(result.scores)  # {"accuracy": 0.95, "latency": 1.2, "my_custom": 0.8}

# Option B: Run workflow + evaluate in one call
result = evaluate(
    workflow="path/to/config.yml",
    dataset=[{"question": "What is 2+3?", "answer": "5"}],
    evaluators=["accuracy"],
)

# Option C: Score a single item
score = evaluate.score(
    question="What is 2+3?",
    expected="5",
    actual="The answer is 5",
    evaluator="accuracy",
)
```

The current API is designed for batch evaluation orchestration (CLI-first).
For the Python API use case (nvagenteval integration, ATIF scoring),
a thinner layer on top would significantly reduce friction.